# Property extraction setup

Prepares `data/polymer_samples.db` and the UID manifests for the property extraction runs.

1. **Current tables** (read-only): what is in the database now.
2. **Create the property tables** (**writes to the database**): `properties` and `property_extraction_progress`.
   Uses `CREATE TABLE IF NOT EXISTS`, so running it again changes nothing.
3. **Create the UID manifests** (**writes files** to `data/uid_manifest/`): every paper flagged by property
   presence for at least one of the configured properties and with at least one sample, `MANIFEST_SIZE` UIDs per file.

The properties and their presence labels come from the agent's `PROPERTY_CONFIG`.

In [1]:
import sys
sys.dont_write_bytecode = True
import json
import random
import sqlite3
from contextlib import closing
from pathlib import Path

import pandas as pd
from IPython.display import display

# Find the repository from this folder or any parent.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "data/polymer_samples.db").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from synth_extract.agents.extraction.property_config import PROPERTY_CONFIG

DB_PATH = ROOT / "data" / "polymer_samples.db"
MANIFEST_DIR = ROOT / "data" / "uid_manifest"
PROPERTIES = list(PROPERTY_CONFIG)     # the properties to extract (keys of PROPERTY_CONFIG)


def read_only():
    return closing(sqlite3.connect(f"{DB_PATH.as_uri()}?mode=ro", uri=True, timeout=60))


def read_write():
    return closing(sqlite3.connect(DB_PATH, timeout=60))


print("Database:", DB_PATH)
print("Properties:", ", ".join(PROPERTIES))

Database: /Users/kevinge/Work/Data Extraction/synth_extract/data/polymer_samples.db
Properties: glass_transition_temperature, melting_temperature, thermal_decomposition_temperature, number_average_molecular_weight, weight_average_molecular_weight, dispersity


## 1. Current tables (read-only)

Every table with its row count and columns.

In [2]:
with read_only() as con:
    tables = [r[0] for r in con.execute(
        "SELECT name FROM sqlite_master WHERE type = 'table' AND name NOT LIKE 'sqlite_%' ORDER BY name")]
    for table in tables:
        n = con.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
        columns = pd.read_sql_query(f"PRAGMA table_info({table})", con)[["name", "type", "notnull", "dflt_value", "pk"]]
        print(f"\n=== {table}: {n:,} rows ===")
        display(columns)
    print("\nforeign_keys:", con.execute("PRAGMA foreign_keys").fetchone()[0],
          "| journal_mode:", con.execute("PRAGMA journal_mode").fetchone()[0])


=== papers: 273,191 rows ===


,name,type,notnull,dflt_value,pk
0,paper_id,INTEGER,0,None,1
1,paper_uid,TEXT,1,None,0
2,identifier_type,TEXT,1,None,0
3,identifier_value,TEXT,1,None,0
4,doi,TEXT,0,None,0
5,arxiv_id,TEXT,0,None,0
6,pmcid,TEXT,0,None,0
7,title,TEXT,0,None,0
8,class_label,INTEGER,0,NULL,0
9,sources,TEXT,1,None,0



=== property_classification: 273,191 rows ===


,name,type,notnull,dflt_value,pk
0,paper_uid,TEXT,0,None,1
1,status,TEXT,1,'pending',0
2,attempt_count,INTEGER,1,0,0
3,has_property,INTEGER,0,None,0
4,properties_json,TEXT,0,None,0
5,last_error,TEXT,0,None,0
6,started_at,TEXT,0,None,0
7,completed_at,TEXT,0,None,0
8,updated_at,TEXT,1,CURRENT_TIMESTAMP,0



=== sample_extraction_progress: 273,191 rows ===


,name,type,notnull,dflt_value,pk
0,paper_uid,TEXT,0,None,1
1,status,TEXT,1,'pending',0
2,attempt_count,INTEGER,1,0,0
3,last_error,TEXT,0,None,0
4,started_at,TEXT,0,None,0
5,completed_at,TEXT,0,None,0
6,updated_at,TEXT,1,CURRENT_TIMESTAMP,0



=== samples: 1,954,131 rows ===


,name,type,notnull,dflt_value,pk
0,sample_id,INTEGER,0,None,1
1,paper_uid,TEXT,1,None,0
2,name,TEXT,1,None,0
3,identifier,TEXT,1,'',0
4,aliases,TEXT,1,'[]',0
5,description,TEXT,1,None,0



foreign_keys: 0 | journal_mode: delete


## 2. Create the property tables

**This cell writes to `data/polymer_samples.db`.** It creates two tables and their indexes if they do not exist;
existing tables and rows are left unchanged.

- `properties`: one row per property value of one sample (the agent's `Property` record). `sample_id` is required;
  `clean` is generated from `flags` (1 when there are no flags). No CHECK lists on `property`, `method`, `unit`, or
  `value_type`, so new properties need no table change; the agent's config and schema enforce the allowed values.
  `ON DELETE CASCADE` removes a sample's properties when the sample is deleted, but only in a connection with
  `PRAGMA foreign_keys = ON`.
- `property_extraction_progress`: run state per (paper, property), like `sample_extraction_progress`, plus the
  prompt (`mode`), prompt hash, model, counts, and tokens of the run that produced the stored rows.

Take a backup first (SQLite backup, not `rsync`) if the database is in use: set `BACKUP_FIRST = True`.

In [3]:
BACKUP_FIRST = False
BACKUP_PATH = ROOT / "data" / "polymer_samples_before_property_tables.db"

PROPERTY_TABLES_DDL = """
CREATE TABLE IF NOT EXISTS properties (
    property_id   INTEGER PRIMARY KEY AUTOINCREMENT,
    property_uid  TEXT GENERATED ALWAYS AS ('PID' || printf('%06d', property_id)) STORED UNIQUE,
    sample_id     INTEGER NOT NULL REFERENCES samples(sample_id) ON DELETE CASCADE,
    paper_uid     TEXT NOT NULL REFERENCES papers(paper_uid),
    property      TEXT NOT NULL,
    method        TEXT NOT NULL,
    subtype       TEXT,
    value_raw     TEXT NOT NULL,
    value         REAL,
    value_type    TEXT NOT NULL,
    unit          TEXT NOT NULL,
    conditions    TEXT NOT NULL DEFAULT '[]' CHECK (json_valid(conditions)),
    origin        TEXT NOT NULL,
    description   TEXT NOT NULL,
    flags         TEXT NOT NULL DEFAULT '[]' CHECK (json_valid(flags)),
    clean         INTEGER GENERATED ALWAYS AS (json_array_length(flags) = 0) STORED
);
CREATE INDEX IF NOT EXISTS idx_properties_sample_id ON properties(sample_id);
CREATE INDEX IF NOT EXISTS idx_properties_paper ON properties(paper_uid, property);
CREATE INDEX IF NOT EXISTS idx_properties_clean ON properties(property, clean);

CREATE TABLE IF NOT EXISTS property_extraction_progress (
    paper_uid         TEXT NOT NULL REFERENCES papers(paper_uid),
    property          TEXT NOT NULL,
    status            TEXT NOT NULL DEFAULT 'pending'
                        CHECK (status IN ('pending', 'in_progress', 'success', 'failed')),
    attempt_count     INTEGER NOT NULL DEFAULT 0 CHECK (attempt_count >= 0),
    mode              TEXT CHECK (mode IS NULL OR mode IN ('per_paper', 'with_context')),
    prompt_hash       TEXT,
    model             TEXT,
    requests          INTEGER,
    records           INTEGER,
    unlinked          INTEGER,
    prompt_tokens     INTEGER,
    completion_tokens INTEGER,
    last_error        TEXT,
    started_at        TEXT,
    completed_at      TEXT,
    updated_at        TEXT NOT NULL DEFAULT CURRENT_TIMESTAMP,
    PRIMARY KEY (paper_uid, property)
);
CREATE INDEX IF NOT EXISTS idx_property_progress_status ON property_extraction_progress(status, property);
"""

if BACKUP_FIRST:
    if BACKUP_PATH.exists():
        raise FileExistsError(f"Backup already exists: {BACKUP_PATH}")
    with read_only() as source, closing(sqlite3.connect(BACKUP_PATH)) as target:
        source.backup(target)
    print("Backup written:", BACKUP_PATH)

with read_write() as con:
    before = {r[0] for r in con.execute("SELECT name FROM sqlite_master WHERE type = 'table'")}
    con.executescript("BEGIN;\n" + PROPERTY_TABLES_DDL + "\nCOMMIT;")
    after = {r[0] for r in con.execute("SELECT name FROM sqlite_master WHERE type = 'table'")}
    for table in ("properties", "property_extraction_progress"):
        state = "created" if table in after - before else "already existed"
        n = con.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
        print(f"{table}: {state}, {n:,} rows")
        print(con.execute("SELECT sql FROM sqlite_master WHERE name = ?", (table,)).fetchone()[0], "\n")

properties: created, 0 rows
CREATE TABLE properties (
    property_id   INTEGER PRIMARY KEY AUTOINCREMENT,
    property_uid  TEXT GENERATED ALWAYS AS ('PID' || printf('%06d', property_id)) STORED UNIQUE,
    sample_id     INTEGER NOT NULL REFERENCES samples(sample_id) ON DELETE CASCADE,
    paper_uid     TEXT NOT NULL REFERENCES papers(paper_uid),
    property      TEXT NOT NULL,
    method        TEXT NOT NULL,
    subtype       TEXT,
    value_raw     TEXT NOT NULL,
    value         REAL,
    value_type    TEXT NOT NULL,
    unit          TEXT NOT NULL,
    conditions    TEXT NOT NULL DEFAULT '[]' CHECK (json_valid(conditions)),
    origin        TEXT NOT NULL,
    description   TEXT NOT NULL,
    flags         TEXT NOT NULL DEFAULT '[]' CHECK (json_valid(flags)),
    clean         INTEGER GENERATED ALWAYS AS (json_array_length(flags) = 0) STORED
) 

property_extraction_progress: created, 0 rows
CREATE TABLE property_extraction_progress (
    paper_uid         TEXT NOT NULL REFERENC

## 3. Create the UID manifests

**This cell writes files** to `data/uid_manifest/`: `property_extraction_uids_0001.txt`, `…_0002.txt`, … with
`MANIFEST_SIZE` UIDs each (the last file has the rest), one UID per line, the same format as the sample manifests.

A paper is included when property presence succeeded and flags at least one property in `PROPERTIES`, and the paper
has at least one sample. With `SHUFFLE = True` the papers are shuffled with a fixed seed before splitting, so large
and small papers spread evenly over the files (the same seed always gives the same files). The table after the
cell shows, per file, the papers, (paper, property) pairs (about the prompt 2 requests; papers with more
than 15 samples need an extra request per batch), samples, and prompt 3 requests (pairs × samples).

Existing property manifests are not overwritten unless `OVERWRITE = True`; sample manifests are never touched.

In [4]:
MANIFEST_SIZE = 1800
MANIFEST_PREFIX = "property_extraction_uids"
SHUFFLE = True
SEED = 0
OVERWRITE = False

labels = {PROPERTY_CONFIG[p]["presence_label"]: p for p in PROPERTIES}
with read_only() as con:
    flags = pd.read_sql_query(
        f"""
        SELECT pc.paper_uid, j.value AS label
        FROM property_classification pc, json_each(pc.properties_json) j
        WHERE pc.status = 'success' AND j.value IN ({", ".join("?" for _ in labels)})
        """,
        con, params=list(labels),
    )
    n_samples = pd.read_sql_query("SELECT paper_uid, COUNT(*) AS samples FROM samples GROUP BY paper_uid", con)

flags = flags.merge(n_samples, on="paper_uid")          # drops papers without samples
flags["property"] = flags.label.map(labels)
papers = (flags.groupby("paper_uid")
          .agg(samples=("samples", "first"), properties=("property", "nunique"))
          .reset_index().sort_values("paper_uid"))
papers["p3_requests"] = papers.properties * papers.samples

uids = papers.paper_uid.tolist()
if SHUFFLE:
    random.Random(SEED).shuffle(uids)
chunks = [uids[i:i + MANIFEST_SIZE] for i in range(0, len(uids), MANIFEST_SIZE)]
print(f"{len(uids):,} papers flagged for at least one of {len(PROPERTIES)} properties and with samples; "
      f"{len(flags):,} (paper, property) pairs; {len(chunks)} manifest(s) of up to {MANIFEST_SIZE}")

MANIFEST_DIR.mkdir(parents=True, exist_ok=True)
existing = sorted(MANIFEST_DIR.glob(f"{MANIFEST_PREFIX}_*.txt"))
if existing and not OVERWRITE:
    raise FileExistsError(f"{len(existing)} property manifest(s) already in {MANIFEST_DIR}; "
                          f"set OVERWRITE = True to replace them.")
for path in existing:
    path.unlink()

by_uid = papers.set_index("paper_uid")
rows = []
for n, chunk in enumerate(chunks, start=1):
    path = MANIFEST_DIR / f"{MANIFEST_PREFIX}_{n:04d}.txt"
    path.write_text("\n".join(chunk) + "\n", encoding="utf-8")
    part = by_uid.loc[chunk]
    rows.append({"manifest": path.name, "papers": len(chunk), "pairs": int(part.properties.sum()),
                 "samples": int(part.samples.sum()), "p3_requests": int(part.p3_requests.sum())})
manifests = pd.DataFrame(rows)
print(f"Wrote {len(chunks)} file(s) to {MANIFEST_DIR}")
with pd.option_context("display.max_rows", 20):
    display(manifests)
print("\nTotals:", manifests[["papers", "pairs", "samples", "p3_requests"]].sum().to_dict())
print("Per file, min / max: pairs", manifests.pairs.min(), "/", manifests.pairs.max(),
      "| p3 requests", manifests.p3_requests.min(), "/", manifests.p3_requests.max())

138,772 papers flagged for at least one of 6 properties and with samples; 278,829 (paper, property) pairs; 78 manifest(s) of up to 1800
Wrote 78 file(s) to /Users/kevinge/Work/Data Extraction/synth_extract/data/uid_manifest


,manifest,papers,pairs,samples,p3_requests
0,property_extraction_uids_0001.txt,1800,3607,14128,30608
1,property_extraction_uids_0002.txt,1800,3658,13699,30450
2,property_extraction_uids_0003.txt,1800,3740,13936,30912
3,property_extraction_uids_0004.txt,1800,3574,14314,30671
4,property_extraction_uids_0005.txt,1800,3621,14546,31871
...,...,...,...,...,...
73,property_extraction_uids_0074.txt,1800,3565,13628,28622
74,property_extraction_uids_0075.txt,1800,3688,14016,30293
75,property_extraction_uids_0076.txt,1800,3642,13938,30353
76,property_extraction_uids_0077.txt,1800,3651,13896,29612



Totals: {'papers': 138772, 'pairs': 278829, 'samples': 1080212, 'p3_requests': 2317433}
Per file, min / max: pairs 330 / 3740 | p3 requests 2781 / 32245


## 4. Fill `property_extraction_progress`

**This cell writes to `data/polymer_samples.db`.** It adds one `pending` row per (paper, property) pair to extract:
the paper's property presence succeeded and flags the property, the property is in `PROPERTIES`, and the paper has at
least one sample (the same selection as the manifests). Existing rows are kept as they are (`INSERT OR IGNORE`), so
running it again, or after adding a property to `PROPERTIES`, only adds the missing pairs. The run columns (`mode`,
`prompt_hash`, `model`, counts, tokens) stay empty until the runner fills them.

In [5]:
labels = {PROPERTY_CONFIG[p]["presence_label"]: p for p in PROPERTIES}
label_case = " ".join(f"WHEN ? THEN ?" for _ in labels)
label_params = [x for label, prop in labels.items() for x in (label, prop)]

with read_write() as con:
    before = con.execute("SELECT COUNT(*) FROM property_extraction_progress").fetchone()[0]
    con.execute("BEGIN IMMEDIATE")
    con.execute(
        f"""
        INSERT OR IGNORE INTO property_extraction_progress (paper_uid, property)
        SELECT DISTINCT pc.paper_uid, CASE j.value {label_case} END
        FROM property_classification pc, json_each(pc.properties_json) j
        WHERE pc.status = 'success'
          AND j.value IN ({", ".join("?" for _ in labels)})
          AND EXISTS (SELECT 1 FROM samples s WHERE s.paper_uid = pc.paper_uid)
        """,
        label_params + list(labels),
    )
    con.commit()
    after = con.execute("SELECT COUNT(*) FROM property_extraction_progress").fetchone()[0]
    summary = pd.read_sql_query(
        "SELECT property, status, COUNT(*) AS pairs FROM property_extraction_progress "
        "GROUP BY property, status ORDER BY property, status", con)

print(f"property_extraction_progress: {before:,} rows before, {after:,} after ({after - before:,} added)")
display(summary)

property_extraction_progress: 0 rows before, 278,829 after (278,829 added)


,property,status,pairs
0,dispersity,pending,48148
1,glass_transition_temperature,pending,50261
2,melting_temperature,pending,31428
3,number_average_molecular_weight,pending,47812
4,thermal_decomposition_temperature,pending,67288
5,weight_average_molecular_weight,pending,33892
